# 01 — Business Problem and Data Understanding

**Goal:** turn the retail forecasting problem into measurable forecasting and inventory questions. This notebook is intentionally analysis-heavy and includes interactive exploration.

In [ ]:

from pathlib import Path
import os, sys
ROOT = Path.cwd()
while not (ROOT / "README.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
print("Project root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display
import ipywidgets as widgets
from src.data.loader import raw_paths
from src.data.validator import validate_raw_files
missing = validate_raw_files(ROOT/"data/raw")
print("Missing raw files:", missing if missing else "None")

### Business framing

We care about two linked decisions:

1. **How much demand should we forecast?**
2. **How should that forecast drive inventory decisions?**

The business tension is between stockouts and excess inventory. Forecast quality is therefore necessary but not sufficient: the final system must connect forecasts to cost and service-level outcomes.

In [ ]:
from src.utils.config import load_yaml
data_cfg=load_yaml("data_config.yaml"); inv_cfg=load_yaml("inventory_config.yaml")
display(pd.DataFrame({"setting":["max_items_per_store","max_total_series","baseline_service_level","default_service_level","default_lead_time_days","default_review_period_days"],"value":[data_cfg["max_items_per_store"],data_cfg["max_total_series"],inv_cfg["baseline_service_level"],inv_cfg["default_service_level"],inv_cfg["default_lead_time_days"],inv_cfg["default_review_period_days"]]}))

In [ ]:
import pandas as pd
path=ROOT/"data/interim/cleaned_sales.parquet"
if path.exists():
    df=pd.read_parquet(path, columns=["id","store_id","item_id","date","demand"]); print(df.shape); print("Series:",df.id.nunique()); print("Date range:",df.date.min(),"to",df.date.max()); display(df.head())
else: print("Run python -m scripts.prepare_data first.")

In [ ]:
def show_series(store_id,item_id):
    p=ROOT/"data/interim/cleaned_sales.parquet"
    if not p.exists(): return print("Run prepare_data first.")
    x=pd.read_parquet(p); q=x[(x.store_id.astype(str)==str(store_id))&(x.item_id.astype(str)==str(item_id))].sort_values("date"); display(q[["date","demand","sell_price"]].tail(28))
if (ROOT/"data/interim/cleaned_sales.parquet").exists():
    tmp=pd.read_parquet(ROOT/"data/interim/cleaned_sales.parquet",columns=["store_id","item_id"]); stores=sorted(tmp.store_id.astype(str).unique()); s=widgets.Dropdown(options=stores,description="Store"); items=widgets.Dropdown(description="Item");
    def refresh(*_):
        vals=sorted(tmp.loc[tmp.store_id.astype(str)==s.value,"item_id"].astype(str).unique()); items.options=vals
    s.observe(refresh,names="value"); refresh(); widgets.interact(show_series,store_id=s,item_id=items)